In [ ]:
import torch
import torch.nn.functional as F

import tonic 
import tonic.transforms as T

import numpy as np
import matplotlib.pyplot as plt

from src.utils.config_utils import setup_config_dictionary
from src.utils.save_utils import interactive_load_model
from src.utils.plot_utils import plot_frames, plot_cams, plot_combined_frames, plot_activations, combine_polarities
from src.utils.data_utils import get_dataloaders
from src.utils.model_utils import get_model

from src.temporal_score_cam import TemporalScoreCAM

config = setup_config_dictionary()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(torch.seed())
# torch.manual_seed(4762538657146009718)

In [ ]:
gesture_map = {
    0: "hand_clapping",
    1: "right_hand_wave",
    2: "left_hand_wave",
    3: "right_hand_clockwise",
    4: "right_hand_counter_clockwise",
    5: "left_hand_clockwise",
    6: "left_hand_counter_clockwise",
    7: "forearm_roll_forward",
    8: "forearm_roll_backward",
    9: "drums",
    10: "guitar",
    #11: "random_other_gestures", # -> not implemented here
}    
# load the model
train_loader, test_loader = get_dataloaders(
    config["DATASET_NAME"], 
    1, 
    False,
)

event_batches, _ = next(iter(train_loader))
num_channels = event_batches.shape[2]

model = get_model(
        config["MODEL_NAME"],
        num_classes=len(train_loader.dataset.classes),
        num_channels=num_channels,
        time_steps=config["N_TIME_BINS"],
        beta=config["BETA"],
        enable_bntt=config["ENABLE_BNTT"],
    ).to(device)

interactive_load_model(model, config["MODEL_SAVE_DIR"], config["MODEL_NAME"])



In [ ]:
model.eval()
# Visualize using our ScoreCAM implementation
score_cam = TemporalScoreCAM(model)
conv_layers = [module for module in model.modules() if isinstance(module, torch.nn.Conv2d)]

target_layer = conv_layers[-1]

with torch.no_grad():
    for i, (event_batches, labels) in enumerate(test_loader):
        event_batches = event_batches.to(device)
        labels = labels.to(device)

        outputs, mem_history = model(event_batches) #outputs are per-time.
        # only take the last time step for classification for the membrane potential-based models
        output_class = F.softmax(outputs[:, -1, :], dim=1).cpu().numpy()
        predicted_label = int(output_class.argmax(axis=1)[0])

        print(f"Output probabilities: {[f'{p:.2%}' for p in output_class[0]]}")
        print(f"Predicted label {predicted_label},real label: {labels.item()}\nPredicted gesture: {gesture_map[predicted_label]},\
               real gesture: {gesture_map[labels.item()]}")
        
        single_input = event_batches[0]
        frames = single_input.cpu().numpy().transpose(0, 2, 3, 1)
        single_input_singleton_batch = single_input.unsqueeze(0)  # Add batch dimension for processing
        combined_frames = combine_polarities(frames)
        plot_frames(frames)
        plot_combined_frames(combined_frames)
        
        # Step through the scorecam parts to ensure everything works
        activations = score_cam.extract_activations(single_input_singleton_batch, target_layer) # (BATCH, TIME, NUM_CHANNELS, HEIGHT, WIDTH)
        upsampled_activations = score_cam.upsample_activations(activations, single_input_singleton_batch, normalize=True)
        print("Activations before normalisation:")
        plot_activations(activations, combined_frames)
        print("Normalised upsampled activations:")
        plot_activations(upsampled_activations, combined_frames)
        cam_weights = score_cam.get_cam_weights(single_input_singleton_batch, activations)
        # get cam weights for predicted class, e.g. first frame
        for t in range(len(frames)):
            predicted_weights = cam_weights[0, t, :, predicted_label]
            print(predicted_weights)

        break